"""
ThonburianTTS Engine — optimized for repeated inference
"""
import logging
from pathlib import Path
from functools import lru_cache

import torch
from cached_path import cached_path
from flowtts.inference import FlowTTSPipeline, ModelConfig, AudioConfig
from transformers import pipeline as hf_pipeline

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)


# ─── Constants ────────────────────────────────────────────────────────────────

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_CHECKPOINT = "hf://ThuraAung1601/E2-F5-TTS/F5_Thai/mega_f5_last.safetensors"
VOCAB_FILE       = "hf://ThuraAung1601/E2-F5-TTS/F5_Thai/mega_vocab.txt"
REF_AUDIO_URL    = "hf://ThuraAung1601/E2-F5-TTS/ref_samples/ref_sample.wav"
ASR_MODEL        = "biodatlab/whisper-th-medium-combined"

OUTPUT_DIR = Path("outputs_f5")
TEMP_DIR   = "temp_f5"


# ─── Singletons (loaded once, reused forever) ─────────────────────────────────

@lru_cache(maxsize=1)
def get_asr_pipeline():
    """Load Thonburian Whisper ASR once and cache it."""
    logger.info("Loading ASR model: %s on %s", ASR_MODEL, DEVICE)
    return hf_pipeline(
        task="automatic-speech-recognition",
        model=ASR_MODEL,
        chunk_length_s=30,
        device=0 if DEVICE == "cuda" else "cpu",
    )


@lru_cache(maxsize=1)
def get_tts_pipeline():
    """Load FlowTTS pipeline once and cache it."""
    logger.info("Loading TTS model on %s", DEVICE)

    model_config = ModelConfig(
        language="th",
        model_type="F5",
        checkpoint=MODEL_CHECKPOINT,
        vocab_file=VOCAB_FILE,
        ode_method="euler",
        use_ema=True,
        vocoder="vocos",
        device=DEVICE,
    )
    audio_config = AudioConfig(
        silence_threshold=-45,
        max_audio_length=20000,
        cfg_strength=2.5,
        nfe_step=32,
        target_rms=0.1,
        cross_fade_duration=0.15,
        speed=1.0,
        min_silence_len=500,
        keep_silence=200,
        seek_step=10,
    )
    return FlowTTSPipeline(
        model_config=model_config,
        audio_config=audio_config,
        temp_dir=TEMP_DIR,
    )


@lru_cache(maxsize=1)
def get_ref_audio():
    """Download reference audio once and cache the local path + transcription."""
    logger.info("Downloading reference audio...")
    ref_path = str(cached_path(REF_AUDIO_URL))
    ref_text = transcribe(ref_path)
    logger.info("Reference text: %s", ref_text)
    return ref_path, ref_text


# ─── Core functions ───────────────────────────────────────────────────────────

def transcribe(audio_file: str, lang: str = "th") -> str:
    """Transcribe audio file using Thonburian Whisper."""
    asr = get_asr_pipeline()
    result = asr(
        audio_file,
        generate_kwargs={"language": f"<|{lang}|>", "task": "transcribe"},
        batch_size=16,
    )
    return result["text"]


def generate(
    text: str,
    output_filename: str = "output.wav",
    speed: float = 1.0,
) -> str:
    """
    Generate Thai speech from text.

    Args:
        text:            Thai text to synthesize
        output_filename: output .wav filename inside OUTPUT_DIR
        speed:           speech speed multiplier (1.0 = normal)

    Returns:
        Path to the generated .wav file
    """
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    tts          = get_tts_pipeline()
    ref_path, ref_text = get_ref_audio()
    output_path  = str(OUTPUT_DIR / output_filename)

    logger.info("Generating: %s", text)
    result = tts(
        text=text,
        ref_voice=ref_path,
        ref_text=ref_text,
        output_file=output_path,
        speed=speed,
        check_duration=True,
    )
    logger.info("Saved → %s", result)
    return result


# ─── Entry point ──────────────────────────────────────────────────────────────

def main():
    texts = [
        "ยินดีที่ได้รู้จักคุณวันนี้อากาศดีมาก",
        "ขอบคุณมากครับสำหรับการต้อนรับ",
        "ระบบนี้สามารถสังเคราะห์เสียงภาษาไทยได้อย่างเป็นธรรมชาติ",
    ]

    for i, text in enumerate(texts):
        try:
            generate(text, output_filename=f"output_{i+1}.wav")
        except Exception as e:
            logger.error("Failed on '%s': %s", text, e)


if __name__ == "__main__":
    main()